# PCA plots to showcase and analyse all trajectories

In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import logging
import importlib

import plot_pca_framework
importlib.reload(plot_pca_framework)

from plot_pca_framework import (
    load_dataset,
    extract_neuron_data,
    extract_group_averaged_data,
    fit_pca,
    project_onto_pca,
    align_pca_signs,
    run_pca,
    slice_epoch,
    slice_window,
    smooth_trajectories,
    get_event_markers,
    analyze_dataset,
    cross_project,
    cross_class_project,
    compute_trajectory_metrics,
    compute_reconstruction_r2,
    compute_subspace_overlap,
    compute_procrustes_distance,
    compute_cross_correlation,
    compute_cross_epoch_r2_matrix,
    plot_1d_pc_timecourses,
    build_overlay_figure,
    plot_scree_comparison,
    plot_speed_profiles,
    plot_cross_epoch_r2_matrix,
    plot_metric_comparison_table,
    EPOCHS,
    # Epoch-specific analysis
    analyze_epoch,
    get_epoch_event_markers,
    save_epoch_trajectories,
    # Analysis functions
    compute_participation_ratio,
    compute_divergence_onset,
    compute_pc_loadings_by_group,
    plot_pc_loadings,
    plot_participation_ratio_comparison,
    plot_divergence_comparison,
    # RSA + Procrustes
    compute_rdm,
    compare_rdms,
    compute_rsa,
    compute_procrustes_comparison,
    plot_rdm,
    plot_rsa_comparison,
    plot_procrustes_comparison,
    # Null models
    null_cross_projection_r2,
    null_separation,
    null_cross_class_r2,
    null_same_neuron_cross_r2,
    null_reward_deflection,
    null_rsa,
    # Private helper for neuron alignment (used in H4 cross-correlation)
    _align_neuron_data,
)

logging.basicConfig(level=logging.WARNING)
%matplotlib inline
plt.rcParams["figure.dpi"] = 100

## Configuration

In [15]:
# Dataset definitions
DATASETS = {
    'SpontFB': {'mat_file': 'dataSpontFB.mat', 'var_name': 'dataSpontFB'},
    'CRFB':    {'mat_file': 'dataCRFB.mat',    'var_name': 'dataCRFB'},
    'ToneFB':  {'mat_file': 'dataToneFB.mat',  'var_name': 'dataToneFB'},
}

# Neuron groups
DA_GROUPS = ['DF', 'DB', 'D', 'DFB']
GABA_GROUPS = ['GF', 'GB', 'G', 'GFB']
ALL_GROUPS = DA_GROUPS + GABA_GROUPS

# Analysis parameters
N_COMPONENTS = 3
EVENT_IDX = 600
WINDOW = 150  # expanded from 120 to capture post-reward dynamics
DT = 0.01
SG_WINDOW = 11
SG_ORDER = 3

## Step 0: Load All Datasets & Run PCA

Run all 9 analyses (3 datasets x {Dopamine, GABA, Combined}). The DFB group has NaN rows in CRFB/ToneFB (rows 19 and 65); these are dropped automatically.

In [16]:
# Run all 9 dataset x neuron-combo analyses
results = {}
combos = {'Dopamine': DA_GROUPS, 'GABA': GABA_GROUPS, 'Combined': ALL_GROUPS}

for ds_name, ds_cfg in DATASETS.items():
    for combo_name, groups in combos.items():
        key = f"{ds_name}_{combo_name}"
        try:
            r = analyze_dataset(
                mat_file=ds_cfg['mat_file'],
                var_name=ds_cfg['var_name'],
                dataset_name=ds_name,
                neuron_groups=groups,
                combo_label=combo_name,
                n_components=N_COMPONENTS,
                event_idx=EVENT_IDX,
                window=WINDOW,
                dt=DT,
                sg_window=SG_WINDOW,
                sg_order=SG_ORDER,
            )
            results[key] = r
            evr = r['explained_variance_ratio']
            print(f"OK  {key:30s}  n={r['n_neurons']:4d}  "
                  f"EVR=[{evr[0]:.3f}, {evr[1]:.3f}, {evr[2]:.3f}]  "
                  f"total={sum(evr):.3f}")
            for g, s in r['stats'].items():
                if s['dropped'] > 0:
                    print(f"     {g}: dropped {s['dropped']}/{s['total']} neurons (NaN)")
        except Exception as e:
            print(f"FAIL {key:30s}  {e}")

print(f"\nSuccessful: {len(results)}/9")

FAIL SpontFB_Dopamine                [Errno 2] No such file or directory: 'dataSpontFB.mat'
FAIL SpontFB_GABA                    [Errno 2] No such file or directory: 'dataSpontFB.mat'
FAIL SpontFB_Combined                [Errno 2] No such file or directory: 'dataSpontFB.mat'
FAIL CRFB_Dopamine                   [Errno 2] No such file or directory: 'dataCRFB.mat'
FAIL CRFB_GABA                       [Errno 2] No such file or directory: 'dataCRFB.mat'
FAIL CRFB_Combined                   [Errno 2] No such file or directory: 'dataCRFB.mat'
FAIL ToneFB_Dopamine                 [Errno 2] No such file or directory: 'dataToneFB.mat'
FAIL ToneFB_GABA                     [Errno 2] No such file or directory: 'dataToneFB.mat'
FAIL ToneFB_Combined                 [Errno 2] No such file or directory: 'dataToneFB.mat'

Successful: 0/9


### H1.1: Same-Neuron Cross-Projection -- CRFB <-> ToneFB

CRFB and ToneFB contain the **same neurons** recorded during the same sessions, but aligned to different events (CR onset vs tone onset). This tests whether the PCA subspace is preserved across alignment conditions.

**Prediction (movement):** CRFB and ToneFB share neurons performing the same F/B movement. If DA encodes movement, their PCA subspaces should overlap strongly and cross-projection R-squared should be high.

**Prediction (RPE):** ToneFB should have additional value-related variance dimensions (CS-locked value signal) not present in CRFB. Subspace overlap should be incomplete, especially for DA.


In [ ]:
# Same-neuron cross-projection: GABA
cross_results_same_neuron = {}

if 'ToneFB_GABA' in results and 'CRFB_GABA' in results:
    cross_results_same_neuron['GABA Tone->CR'] = cross_project(
        results['ToneFB_GABA'], results['CRFB_GABA'],
        use_group_avg=False, window=WINDOW, event_idx=EVENT_IDX,
        dt=DT, sg_window=SG_WINDOW, sg_order=SG_ORDER,
    )
    cross_results_same_neuron['GABA CR->Tone'] = cross_project(
        results['CRFB_GABA'], results['ToneFB_GABA'],
        use_group_avg=False, window=WINDOW, event_idx=EVENT_IDX,
        dt=DT, sg_window=SG_WINDOW, sg_order=SG_ORDER,
    )
    for name, cr in cross_results_same_neuron.items():
        print(f"{name}:  R-squared = {cr['r2']:.4f}")

    # Use aligned PCAs from cross_project (same neuron intersection)
    overlap_gaba = compute_subspace_overlap(
        cross_results_same_neuron['GABA Tone->CR']['pca_fit'].components_,
        cross_results_same_neuron['GABA CR->Tone']['pca_fit'].components_,
    )
    print(f"\nGABA subspace overlap (cos principal angles): {overlap_gaba}")
else:
    print('ToneFB_GABA or CRFB_GABA not available')

# Same-neuron cross-projection: DA
if 'ToneFB_Dopamine' in results and 'CRFB_Dopamine' in results:
    cross_results_same_neuron['DA Tone->CR'] = cross_project(
        results['ToneFB_Dopamine'], results['CRFB_Dopamine'],
        use_group_avg=False, window=WINDOW, event_idx=EVENT_IDX,
        dt=DT, sg_window=SG_WINDOW, sg_order=SG_ORDER,
    )
    cross_results_same_neuron['DA CR->Tone'] = cross_project(
        results['CRFB_Dopamine'], results['ToneFB_Dopamine'],
        use_group_avg=False, window=WINDOW, event_idx=EVENT_IDX,
        dt=DT, sg_window=SG_WINDOW, sg_order=SG_ORDER,
    )
    for name, cr in cross_results_same_neuron.items():
        if 'DA' in name:
            print(f"{name}:  R-squared = {cr['r2']:.4f}")

    # Use aligned PCAs from cross_project (same neuron intersection)
    overlap_da = compute_subspace_overlap(
        cross_results_same_neuron['DA Tone->CR']['pca_fit'].components_,
        cross_results_same_neuron['DA CR->Tone']['pca_fit'].components_,
    )
    print(f"\nDA subspace overlap (cos principal angles): {overlap_da}")
else:
    print('ToneFB_Dopamine or CRFB_Dopamine not available')

ToneFB_GABA or CRFB_GABA not available
ToneFB_Dopamine or CRFB_Dopamine not available


### H1.2: Group-Averaged Cross-Projection -- SpontFB <-> Task Data

SpontFB uses different recording sessions (different individual neurons) but the same neuron classes. We average firing rates within each selectivity group (DF, DB, D, DFB or GF, GB, G, GFB) to create 4 "pseudo-neurons," enabling cross-projection despite non-overlapping neuron IDs.

**Prediction (movement):** If direction encoding is the dominant latent variable, group-averaged SpontFB PCs should capture Task data variance well, even across different neurons. This would show that movement-direction encoding is a population-level property, not dependent on specific neurons.

**Prediction (RPE):** If DA carries value signals in addition to movement, SpontFB PCs (no reward context) should capture less DA variance from task data than GABA variance. GABA R-squared > DA R-squared would suggest DA has extra non-movement components.

**Caveat:** R-squared is structurally biased upward with 4 features and 3 PCs. The null model addresses this.


In [ ]:
# GABA group-averaged cross-projection: SpontFB <-> Task
cross_results_group_avg = {}

for target_name in ['CRFB_GABA', 'ToneFB_GABA']:
    if 'SpontFB_GABA' in results and target_name in results:
        key = f'Spont->{target_name.split("_")[0]}'
        cross_results_group_avg[key] = cross_project(
            results['SpontFB_GABA'], results[target_name],
            use_group_avg=True, neuron_groups=GABA_GROUPS,
            window=WINDOW, event_idx=EVENT_IDX, dt=DT,
            sg_window=SG_WINDOW, sg_order=SG_ORDER,
        )
        print(f"GABA {key}: R-squared = {cross_results_group_avg[key]['r2']:.4f}")

for source_name in ['CRFB_GABA', 'ToneFB_GABA']:
    if source_name in results and 'SpontFB_GABA' in results:
        key = f'{source_name.split("_")[0]}->Spont'
        cross_results_group_avg[key] = cross_project(
            results[source_name], results['SpontFB_GABA'],
            use_group_avg=True, neuron_groups=GABA_GROUPS,
            window=WINDOW, event_idx=EVENT_IDX, dt=DT,
            sg_window=SG_WINDOW, sg_order=SG_ORDER,
        )
        print(f"GABA {key}: R-squared = {cross_results_group_avg[key]['r2']:.4f}")

GABA Spont->CRFB: R-squared = 0.9234
GABA Spont->ToneFB: R-squared = 0.9305
GABA CRFB->Spont: R-squared = 0.9338
GABA ToneFB->Spont: R-squared = 0.9241


In [ ]:
# DA group-averaged cross-projection: SpontFB <-> Task
for target_name in ['CRFB_Dopamine', 'ToneFB_Dopamine']:
    if 'SpontFB_Dopamine' in results and target_name in results:
        key = f'DA Spont->{target_name.split("_")[0]}'
        cr = cross_project(
            results['SpontFB_Dopamine'], results[target_name],
            use_group_avg=True, neuron_groups=DA_GROUPS,
            window=WINDOW, event_idx=EVENT_IDX, dt=DT,
            sg_window=SG_WINDOW, sg_order=SG_ORDER,
        )
        cross_results_group_avg[key] = cr
        print(f"{key}: R-squared = {cr['r2']:.4f}")

for source_name in ['CRFB_Dopamine', 'ToneFB_Dopamine']:
    if source_name in results and 'SpontFB_Dopamine' in results:
        key = f'DA {source_name.split("_")[0]}->Spont'
        cr = cross_project(
            results[source_name], results['SpontFB_Dopamine'],
            use_group_avg=True, neuron_groups=DA_GROUPS,
            window=WINDOW, event_idx=EVENT_IDX, dt=DT,
            sg_window=SG_WINDOW, sg_order=SG_ORDER,
        )
        cross_results_group_avg[key] = cr
        print(f"{key}: R-squared = {cr['r2']:.4f}")

DA Spont->CRFB: R-squared = 0.9278
DA Spont->ToneFB: R-squared = 0.9438
DA CRFB->Spont: R-squared = 0.8948
DA ToneFB->Spont: R-squared = 0.9007


### H1.3: Overlay Visualisations

**Prediction:** Overlaid Task trajectories on SpontFB PCA space should follow similar F/B-separated paths, because both are driven by the same movement. If Task trajectories diverge from SpontFB trajectories, task-specific signals (CS, reward) drive additional variance not present during spontaneous movement.

Visual inspection: do the projected Task trajectories maintain the same fwd/bwd separation pattern as SpontFB?


In [18]:
# Overlay: ToneFB native + CRFB projected (GABA)
if 'GABA Tone->CR' in cross_results_same_neuron and 'ToneFB_GABA' in results:
    tsets = [
        {
            'fwd_smooth': results['ToneFB_GABA']['smooth_data']['fwd_smooth'],
            'bwd_smooth': results['ToneFB_GABA']['smooth_data']['bwd_smooth'],
            'label': 'ToneFB (native)',
            'fwd_color': 'orangered', 'bwd_color': 'royalblue', 'dash': 'solid',
            'event_markers': results['ToneFB_GABA']['event_markers'],
        },
        {
            'fwd_smooth': cross_results_same_neuron['GABA Tone->CR']['smooth_data']['fwd_smooth'],
            'bwd_smooth': cross_results_same_neuron['GABA Tone->CR']['smooth_data']['bwd_smooth'],
            'label': 'CRFB (projected)',
            'fwd_color': 'darkorange', 'bwd_color': 'steelblue', 'dash': 'dash',
            'event_markers': cross_results_same_neuron['GABA Tone->CR']['event_markers'],
        },
    ]
    fig_overlay = build_overlay_figure(
        tsets, title='H1: ToneFB vs CRFB GABA (ToneFB PC space)')
    fig_overlay.show()

In [19]:
# Overlay: SpontFB (self-projected) + Task projected (GABA, group-averaged)
# Both trajectories are in the same group-averaged PCA space for valid comparison.
if 'SpontFB_GABA' in results:
    tsets = []

    # Use the first available cross-projection's PCA to self-project SpontFB
    first_xp_key = None
    for k in ['Spont->CRFB', 'Spont->ToneFB']:
        if k in cross_results_group_avg:
            first_xp_key = k
            break

    if first_xp_key is not None:
        pca_grp = cross_results_group_avg[first_xp_key]['pca_fit']
        X_spont_avg, ts_spont, _ = extract_group_averaged_data(
            results['SpontFB_GABA']['data'], GABA_GROUPS)
        proj_spont = project_onto_pca(pca_grp, X_spont_avg)
        win_spont = slice_window(proj_spont, ts_spont, EVENT_IDX, WINDOW, DT)
        sd_spont = smooth_trajectories(win_spont, SG_WINDOW, SG_ORDER)
        tsets.append({
            'fwd_smooth': sd_spont['fwd_smooth'],
            'bwd_smooth': sd_spont['bwd_smooth'],
            'label': 'SpontFB GABA (self-proj, group-avg PCA)',
            'fwd_color': 'orangered', 'bwd_color': 'royalblue', 'dash': 'solid',
            'event_markers': results['SpontFB_GABA']['event_markers'],
        })

    for key, color_fwd, color_bwd in [
        ('Spont->CRFB', 'gold', 'teal'),
        ('Spont->ToneFB', 'lime', 'purple'),
    ]:
        if key in cross_results_group_avg:
            cr = cross_results_group_avg[key]
            tsets.append({
                'fwd_smooth': cr['smooth_data']['fwd_smooth'],
                'bwd_smooth': cr['smooth_data']['bwd_smooth'],
                'label': f'{cr['project_dataset']} GABA (proj)',
                'fwd_color': color_fwd, 'bwd_color': color_bwd, 'dash': 'dash',
                'event_markers': cr['event_markers'],
            })
    if len(tsets) > 1:
        fig = build_overlay_figure(
            tsets, title='H1: SpontFB vs Task GABA (group-avg SpontFB PC space)')
        fig.show()


In [20]:
# Overlay: SpontFB (self-projected) + Task projected (Dopamine, group-averaged)
# Both trajectories are in the same group-averaged PCA space for valid comparison.
if 'SpontFB_Dopamine' in results:
    tsets = []

    # Use DA cross-projection's PCA to self-project SpontFB
    first_xp_key = None
    for k in ['DA Spont->CRFB', 'DA Spont->ToneFB']:
        if k in cross_results_group_avg:
            first_xp_key = k
            break

    if first_xp_key is not None:
        pca_grp = cross_results_group_avg[first_xp_key]['pca_fit']
        X_spont_avg, ts_spont, _ = extract_group_averaged_data(
            results['SpontFB_Dopamine']['data'], DA_GROUPS)
        proj_spont = project_onto_pca(pca_grp, X_spont_avg)
        win_spont = slice_window(proj_spont, ts_spont, EVENT_IDX, WINDOW, DT)
        sd_spont = smooth_trajectories(win_spont, SG_WINDOW, SG_ORDER)
        tsets.append({
            'fwd_smooth': sd_spont['fwd_smooth'],
            'bwd_smooth': sd_spont['bwd_smooth'],
            'label': 'SpontFB Dopamine (self-proj, group-avg PCA)',
            'fwd_color': 'orangered', 'bwd_color': 'royalblue', 'dash': 'solid',
            'event_markers': results['SpontFB_Dopamine']['event_markers'],
        })

    for key, color_fwd, color_bwd in [
        ('DA Spont->CRFB', 'gold', 'teal'),
        ('DA Spont->ToneFB', 'lime', 'purple'),
    ]:
        if key in cross_results_group_avg:
            cr = cross_results_group_avg[key]
            tsets.append({
                'fwd_smooth': cr['smooth_data']['fwd_smooth'],
                'bwd_smooth': cr['smooth_data']['bwd_smooth'],
                'label': f'{cr['project_dataset']} Dopamine (proj)',
                'fwd_color': color_fwd, 'bwd_color': color_bwd, 'dash': 'dash',
                'event_markers': cr['event_markers'],
            })
    if len(tsets) > 1:
        fig = build_overlay_figure(
            tsets, title='H1: SpontFB vs Task Dopamine (group-avg SpontFB PC space)')
        fig.show()


### H2.5: Per-Neuron-Class PCA

Run PCA on individual neuron classes (DF, DB, D, DFB, GF, GB, G, GFB) across all 3 datasets.

**Key predictions:**
- **DF/DB:** Expect distinct, opposite trajectories for F vs B movements in SpontFB and CRFB. DF neurons should have fwd-dominant PC excursions; DB neurons bwd-dominant. If both show strong separation, DA encodes movement direction at the single-class level.
- **GF/GB:** Expect direction selectivity similar to DF/DB. At reward time (ToneFB, t=700), expect **NO trajectory change** -- if GF/GB are pure movement encoders, reward delivery should not alter their trajectories.
- **GFB:** Test if bidirectional GABA neurons are sensitive to movement direction despite responding to both directions. Some direction sensitivity in GFB would strengthen the movement-encoding interpretation.
- **D/DFB:** Non-selective and bidirectional DA. If even these show some F/B separation, direction encoding is pervasive in DA.

**Cross-class projections:** Fit PCA on class A, project class B. High R-squared = both classes capture the same latent direction variable.


In [21]:
# Per-neuron-class PCA: run each class individually on all 3 datasets
single_class_results = {}

for ds_name, ds_cfg in DATASETS.items():
    for group in ALL_GROUPS:
        key = f"{ds_name}_{group}"
        try:
            r = analyze_dataset(
                mat_file=ds_cfg['mat_file'],
                var_name=ds_cfg['var_name'],
                dataset_name=ds_name,
                neuron_groups=[group],
                combo_label=group,
                n_components=N_COMPONENTS,
                event_idx=EVENT_IDX,
                window=WINDOW,
                dt=DT,
                sg_window=SG_WINDOW,
                sg_order=SG_ORDER,
            )
            single_class_results[key] = r
            evr = r['explained_variance_ratio']
            evr_str = '+'.join(f'{v:.3f}' for v in evr)
            sep = r['metrics']['mean_separation']
            print(f"OK  {key:25s}  n={r['n_neurons']:4d}  EVR=[{evr_str}]  sep={sep:.2f}")
        except Exception as e:
            print(f"SKIP {key:25s}  {e}")

print(f"\nSuccessful single-class analyses: {len(single_class_results)}")

SKIP SpontFB_DF                 [Errno 2] No such file or directory: 'dataSpontFB.mat'
SKIP SpontFB_DB                 [Errno 2] No such file or directory: 'dataSpontFB.mat'
SKIP SpontFB_D                  [Errno 2] No such file or directory: 'dataSpontFB.mat'
SKIP SpontFB_DFB                [Errno 2] No such file or directory: 'dataSpontFB.mat'
SKIP SpontFB_GF                 [Errno 2] No such file or directory: 'dataSpontFB.mat'
SKIP SpontFB_GB                 [Errno 2] No such file or directory: 'dataSpontFB.mat'
SKIP SpontFB_G                  [Errno 2] No such file or directory: 'dataSpontFB.mat'
SKIP SpontFB_GFB                [Errno 2] No such file or directory: 'dataSpontFB.mat'
SKIP CRFB_DF                    [Errno 2] No such file or directory: 'dataCRFB.mat'
SKIP CRFB_DB                    [Errno 2] No such file or directory: 'dataCRFB.mat'
SKIP CRFB_D                     [Errno 2] No such file or directory: 'dataCRFB.mat'
SKIP CRFB_DFB                   [Errno 2] No such fi

In [22]:
# Cross-class projection: fit PCA on class A, project class B
selective_results = {}

for class_a, class_b in [('DF', 'DB'), ('GF', 'GB')]:
    key_a = f'SpontFB_{class_a}'
    key_b = f'SpontFB_{class_b}'
    if key_a not in single_class_results or key_b not in single_class_results:
        print(f'SKIP {class_a}->{class_b}: missing single-class result')
        continue
    try:
        r = cross_class_project(
            result_a=single_class_results[key_a],
            result_b=single_class_results[key_b],
            window=WINDOW, event_idx=EVENT_IDX, dt=DT,
            sg_window=SG_WINDOW, sg_order=SG_ORDER,
        )
        selective_results[f'{class_a}->{class_b}'] = r
        r2_str = '  '.join(f'PC{k+1}={v:.3f}' for k, v in enumerate(r['r2_per_pc']))
        print(f'OK  {class_a}->{class_b}  mean R2={r["r2_train"]:.4f}   {r2_str}')
    except Exception as e:
        print(f'SKIP {class_a}->{class_b}: {e}')

SKIP DF->DB: missing single-class result
SKIP GF->GB: missing single-class result


In [23]:
# Overlay: class A native + class B projected
# Debug: print neuron counts per class
print("Neuron counts per class (SpontFB):")
for g in ALL_GROUPS:
    k = f"SpontFB_{g}"
    if k in single_class_results:
        print(f"  {g}: {single_class_results[k]['n_neurons']} neurons")
    else:
        print(f"  {g}: MISSING")
print()

for class_a, class_b in [('DF', 'DB'), ('GF', 'GB')]:
    key_native = f"SpontFB_{class_a}"
    key_cross = f"{class_a}->{class_b}"
    if key_native not in single_class_results:
        print(f"SKIP overlay {class_a}/{class_b}: {key_native} not in single_class_results")
        continue
    if key_cross not in selective_results:
        print(f"SKIP overlay {class_a}/{class_b}: {key_cross} not in selective_results")
        continue
    rn = single_class_results[key_native]
    rc = selective_results[key_cross]
    tsets = [
        {
            'fwd_smooth': rn['smooth_data']['fwd_smooth'],
            'bwd_smooth': rn['smooth_data']['bwd_smooth'],
            'label': f'{class_a} (native, own space)',
            'fwd_color': 'orangered', 'bwd_color': 'royalblue', 'dash': 'solid',
            'event_markers': rn['event_markers'],
        },
        {
            'fwd_smooth': rc['smooth_data']['fwd_smooth'],
            'bwd_smooth': rc['smooth_data']['bwd_smooth'],
            'label': f'{class_b} (projected into {class_a} space)',
            'fwd_color': 'darkorange', 'bwd_color': 'steelblue', 'dash': 'dash',
            'event_markers': rc['event_markers'],
        },
    ]
    r2_train = rc.get('r2_train', float('nan'))
    r2_cv = rc.get('r2_cv', float('nan'))
    fig = build_overlay_figure(
        tsets,
        title=f'H2: {class_b} onto {class_a} PC space (SpontFB) | R2_train={r2_train:.3f}, R2_cv={r2_cv:.3f}')
    fig.show()


Neuron counts per class (SpontFB):
  DF: MISSING
  DB: MISSING
  D: MISSING
  DFB: MISSING
  GF: MISSING
  GB: MISSING
  G: MISSING
  GFB: MISSING

SKIP overlay DF/DB: SpontFB_DF not in single_class_results
SKIP overlay GF/GB: SpontFB_GF not in single_class_results


### H4.2: Overlay -- ToneFB vs CRFB Dynamics

In [24]:
# Overlay: ToneFB native + CRFB projected for DA
if 'DA Tone->CR' in cross_results_same_neuron and 'ToneFB_Dopamine' in results:
    tsets = [
        {
            'fwd_smooth': results['ToneFB_Dopamine']['smooth_data']['fwd_smooth'],
            'bwd_smooth': results['ToneFB_Dopamine']['smooth_data']['bwd_smooth'],
            'label': 'ToneFB DA (native)',
            'fwd_color': 'orangered', 'bwd_color': 'royalblue', 'dash': 'solid',
            'event_markers': results['ToneFB_Dopamine']['event_markers'],
        },
        {
            'fwd_smooth': cross_results_same_neuron['DA Tone->CR']['smooth_data']['fwd_smooth'],
            'bwd_smooth': cross_results_same_neuron['DA Tone->CR']['smooth_data']['bwd_smooth'],
            'label': 'CRFB DA (projected)',
            'fwd_color': 'darkorange', 'bwd_color': 'steelblue', 'dash': 'dash',
            'event_markers': cross_results_same_neuron['DA Tone->CR']['event_markers'],
        },
    ]
    fig_overlay = build_overlay_figure(
        tsets, title='H4: ToneFB vs CRFB Dopamine (ToneFB PC space)')
    fig_overlay.show()

---
# H5: Per-Subclass PCA & Cross-Class Projections

**Goal:** Run PCA on each individual neuron class, visualise trajectories, and systematically test cross-class projections.

**Key predictions:**
- **DF on forward trials:** expect strong fwd/bwd separability. If DF neurons encode forward movement, their fwd trajectory should dominate PC1.
- **DB on backward trials:** mirror image of DF -- bwd trajectory dominates.
- **GF/GB at reward (ToneFB):** expect NO reward deflection at t=700. Pure direction encoders should be blind to reward.
- **GFB:** test whether bidirectional GABA shows any direction sensitivity.
- **Cross-class R-squared heatmap:** high values between direction-matched classes (DF-DB, GF-GB) = shared direction subspace. Low values between DA-GABA pairs = different encoding.


### H5.1: Per-Subclass PCA -- Save Plots & Summary


In [25]:
# Save per-class trajectory plots and print summary
import os

for ds_name in ["SpontFB", "CRFB", "ToneFB"]:
    out_dir = f"outputs/{ds_name}/classes"
    os.makedirs(out_dir, exist_ok=True)

print(f"{'Key':30s} {'n':>5s} {'PC1':>6s} {'PC1+2+3':>8s} {'Sep':>6s} {'PostSep':>7s}")
print("-" * 70)

for key in sorted(single_class_results.keys()):
    r = single_class_results[key]
    evr = r["explained_variance_ratio"]
    m = r["metrics"]
    post_sep = m.get("post_event_mean_separation", float("nan"))
    print(f"{key:30s} {r['n_neurons']:5d} {evr[0]:6.3f} {sum(evr):8.3f} "
          f"{m['mean_separation']:6.2f} {post_sep:7.2f}")

    # Save scatter and trajectory plots
    ds = key.split("_")[0]
    cls = "_".join(key.split("_")[1:])
    out_dir = f"outputs/{ds}/classes"

    # Scatter plot
    fig_s, ax_s = plt.subplots(figsize=(8, 6))
    proj = r["projections"]
    n_t = r["timesteps"]
    ax_s.scatter(proj[0, :n_t], proj[1, :n_t], c="orangered", s=2, alpha=0.5, label="fwd")
    ax_s.scatter(proj[0, n_t:], proj[1, n_t:], c="royalblue", s=2, alpha=0.5, label="bwd")
    ax_s.set_xlabel("PC1"); ax_s.set_ylabel("PC2")
    ax_s.set_title(f"{key} scatter (EVR: {evr[0]:.3f}, {evr[1]:.3f})")
    ax_s.legend()
    fig_s.savefig(f"{out_dir}/{key}_scatter.png", dpi=100, bbox_inches="tight")
    plt.close(fig_s)

    # Trajectory plot
    sd = r["smooth_data"]
    fig_t = plt.figure(figsize=(10, 7))
    ax_t = fig_t.add_subplot(111, projection="3d")
    ax_t.plot(*sd["fwd_smooth"], c="orangered", label="fwd")
    ax_t.plot(*sd["bwd_smooth"], c="royalblue", label="bwd")
    ax_t.set_xlabel("PC1"); ax_t.set_ylabel("PC2"); ax_t.set_zlabel("PC3")
    ax_t.set_title(f"{key} trajectory")
    ax_t.legend()
    fig_t.savefig(f"{out_dir}/{key}_trajectory.png", dpi=100, bbox_inches="tight")
    plt.close(fig_t)

print(f"\nPlots saved to outputs/{{dataset}}/classes/")


Key                                n    PC1  PC1+2+3    Sep PostSep
----------------------------------------------------------------------

Plots saved to outputs/{dataset}/classes/


In [26]:
# DF (forward-selective DA): expect strong fwd/bwd separability
for ds_name in ["SpontFB", "CRFB", "ToneFB"]:
    key = f"{ds_name}_DF"
    if key not in single_class_results:
        continue
    r = single_class_results[key]
    em = list(r["event_markers"])
    if ds_name == "ToneFB":
        # Add 'idx' key for the reward event, 100 time points after EVENT_IDX (600)
        # which corresponds to index 150 + 100 = 250 in a 301-point window.
        em = em + [{"time": 1.0, "label": "Reward", "color": "gold", "ls": "--", "idx": 250}]
    fig = plot_1d_pc_timecourses(
        r["window_data"], r["smooth_data"], em,
        title=f"DF 1D timecourses ({ds_name}) -- expect fwd/bwd separation")
    plt.show()

# GFB (GABA bidirectional) at reward: expect NO deflection
for ds_name in ["ToneFB", "SpontFB"]:
    key = f"{ds_name}_GFB"
    if key not in single_class_results:
        continue
    r = single_class_results[key]
    em = list(r["event_markers"])
    if ds_name == "ToneFB":
        # Add 'idx' key for the reward event, 100 time points after EVENT_IDX (600)
        # which corresponds to index 150 + 100 = 250 in a 301-point window.
        em = em + [{"time": 1.0, "label": "Reward", "color": "gold", "ls": "--", "idx": 250}]
    fig = plot_1d_pc_timecourses(
        r["window_data"], r["smooth_data"], em,
        title=f"GFB 1D timecourses ({ds_name}) -- expect NO reward deflection")
    plt.show()

### H5.3: Cross-Class Projections -- All Combinations

Fit PCA on class A, project class B. Present CV R-squared as heatmap.


In [27]:
# Cross-class projection for all combinations (SpontFB)
import os
os.makedirs("outputs/SpontFB/combinations", exist_ok=True)

cross_pairs = [
    ("DF", "DB"), ("DB", "DF"),
    ("GF", "GB"), ("GB", "GF"),
    ("DF", "GF"), ("GF", "DF"),
    ("DB", "GB"), ("GB", "DB"),
    ("GFB", "GF"), ("GF", "GFB"),
    ("DFB", "DF"), ("DF", "DFB"),
]

all_cross_results = {}
for class_a, class_b in cross_pairs:
    key_a = f"SpontFB_{class_a}"
    key_b = f"SpontFB_{class_b}"
    if key_a not in single_class_results or key_b not in single_class_results:
        print(f"SKIP {class_a}->{class_b}: missing data")
        continue
    try:
        r = cross_class_project(
            result_a=single_class_results[key_a],
            result_b=single_class_results[key_b],
            window=WINDOW, event_idx=EVENT_IDX, dt=DT,
            sg_window=SG_WINDOW, sg_order=SG_ORDER,
        )
        all_cross_results[f"{class_a}->{class_b}"] = r
        r2_cv = r.get("r2_cv", float("nan"))
        print(f"OK  {class_a}->{class_b}  R2_train={r['r2_train']:.4f}  R2_cv={r2_cv:.4f}")
    except Exception as e:
        print(f"SKIP {class_a}->{class_b}: {e}")


SKIP DF->DB: missing data
SKIP DB->DF: missing data
SKIP GF->GB: missing data
SKIP GB->GF: missing data
SKIP DF->GF: missing data
SKIP GF->DF: missing data
SKIP DB->GB: missing data
SKIP GB->DB: missing data
SKIP GFB->GF: missing data
SKIP GF->GFB: missing data
SKIP DFB->DF: missing data
SKIP DF->DFB: missing data
